# 05 Train Video Model: CNN + LSTM

Academic implementation of video deepfake detection using EfficientNet-B0 for per-frame spatial feature extraction and an LSTM for temporal sequence modeling.

> This CNN+LSTM model is implemented for academic purposes.
> For deployment, we use a frame-based approach using the image model for efficiency and simplicity.

## Step 1: Setup

The notebook assumes the following future dataset layout:

```text
data/splits/videos/
    train/
        real/
        fake/
    val/
        real/
        fake/
    test/
        real/
        fake/
```

In [ ]:
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm.auto import tqdm

DATA_ROOT = Path("../data/splits/videos")
MODEL_DIR = Path("../models/video")
MODEL_PATH = MODEL_DIR / "cnn_lstm_model.pth"

SPLITS = ["train", "val", "test"]
CLASSES = ["real", "fake"]
LABEL_MAP = {"real": 0, "fake": 1}
VIDEO_EXTENSIONS = {".mp4", ".avi", ".mov", ".mkv", ".webm"}

SEQUENCE_LENGTH = 20
IMAGE_SIZE = 224
BATCH_SIZE = 2
NUM_WORKERS = 0
EPOCHS = 5
LEARNING_RATE = 1e-4
LSTM_HIDDEN_SIZE = 256
LSTM_NUM_LAYERS = 1
FREEZE_CNN = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Step 1: Data Loading

Collect video file paths into a pandas DataFrame and inspect the first rows plus class distribution.

In [ ]:
def build_video_dataframe(data_root: Path) -> pd.DataFrame:
    records = []

    for split in SPLITS:
        for class_name in CLASSES:
            class_dir = data_root / split / class_name
            print(f"Scanning {class_dir} | exists={class_dir.exists()}")

            if not class_dir.exists():
                continue

            for video_path in sorted(class_dir.rglob("*")):
                if video_path.suffix.lower() in VIDEO_EXTENSIONS:
                    records.append(
                        {
                            "video_path": str(video_path),
                            "split": split,
                            "label_name": class_name,
                            "label": LABEL_MAP[class_name],
                        }
                    )

    return pd.DataFrame(records)


df = build_video_dataframe(DATA_ROOT)
print(f"Loaded records: {len(df)}")
df.head()

In [ ]:
if len(df) > 0:
    display(pd.crosstab(df["split"], df["label_name"], margins=True))
else:
    print("No videos found yet. Add files under ../data/splits/videos before training.")

## Step 2: Frame Extraction

Each video is sampled to a fixed number of frames, resized to 224 x 224, converted to tensors, and normalized with ImageNet mean and standard deviation.

In [ ]:
frame_transform = transforms.Compose(
    [
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ]
)


def extract_frames(video_path: str, sequence_length: int = SEQUENCE_LENGTH, transform=frame_transform) -> torch.Tensor:
    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frames = []

    if total_frames > 0:
        frame_indices = np.linspace(0, max(total_frames - 1, 0), sequence_length).astype(int)
    else:
        frame_indices = np.arange(sequence_length)

    for frame_index in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_index))
        success, frame = cap.read()

        if not success:
            continue

        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = Image.fromarray(frame)
        frames.append(transform(frame))

    cap.release()

    if len(frames) == 0:
        frames = [torch.zeros(3, IMAGE_SIZE, IMAGE_SIZE)]

    while len(frames) < sequence_length:
        frames.append(frames[-1].clone())

    frames = frames[:sequence_length]
    return torch.stack(frames)  # (sequence_length, 3, 224, 224)

## Step 3: PyTorch Dataset

The dataset returns `(sequence_of_frames, label)` where the video tensor shape is `(sequence_length, 3, 224, 224)`.

In [ ]:
class DeepfakeVideoDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, sequence_length: int = SEQUENCE_LENGTH, transform=frame_transform):
        self.dataframe = dataframe.reset_index(drop=True)
        self.sequence_length = sequence_length
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        frames = extract_frames(row["video_path"], self.sequence_length, self.transform)
        label = torch.tensor(row["label"], dtype=torch.float32)
        return frames, label


train_df = df[df["split"] == "train"].copy() if len(df) > 0 else pd.DataFrame()
val_df = df[df["split"] == "val"].copy() if len(df) > 0 else pd.DataFrame()
test_df = df[df["split"] == "test"].copy() if len(df) > 0 else pd.DataFrame()

train_dataset = DeepfakeVideoDataset(train_df)
val_dataset = DeepfakeVideoDataset(val_df)
test_dataset = DeepfakeVideoDataset(test_df)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(f"Train videos: {len(train_dataset)}")
print(f"Val videos: {len(val_dataset)}")
print(f"Test videos: {len(test_dataset)}")

## Step 4: Model Architecture

EfficientNet-B0 extracts a feature vector for each frame. The LSTM receives the frame feature sequence and produces a binary deepfake classification logit.

In [ ]:
class CNNLSTMDeepfakeDetector(nn.Module):
    def __init__(
        self,
        hidden_size: int = LSTM_HIDDEN_SIZE,
        num_layers: int = LSTM_NUM_LAYERS,
        freeze_cnn: bool = FREEZE_CNN,
    ):
        super().__init__()

        weights = models.EfficientNet_B0_Weights.DEFAULT
        self.cnn = models.efficientnet_b0(weights=weights)
        self.feature_dim = self.cnn.classifier[1].in_features
        self.cnn.classifier = nn.Identity()

        self.freeze_cnn = freeze_cnn

        if self.freeze_cnn:
            for parameter in self.cnn.parameters():
                parameter.requires_grad = False

        self.lstm = nn.LSTM(
            input_size=self.feature_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(hidden_size, 1),
        )

    def forward(self, video_batch: torch.Tensor, debug: bool = False):
        batch_size, sequence_length, channels, height, width = video_batch.shape

        frames = video_batch.reshape(batch_size * sequence_length, channels, height, width)

        if self.freeze_cnn:
            with torch.no_grad():
                frame_features = self.cnn(frames)
        else:
            frame_features = self.cnn(frames)

        feature_sequence = frame_features.reshape(batch_size, sequence_length, self.feature_dim)
        lstm_output, _ = self.lstm(feature_sequence)
        final_timestep = lstm_output[:, -1, :]
        logits = self.classifier(final_timestep).squeeze(1)

        if debug:
            print(f"Batch shape (video tensor): {video_batch.shape}")
            print(f"Feature shape after CNN: {frame_features.shape}")
            print(f"Feature sequence shape: {feature_sequence.shape}")
            print(f"LSTM output shape: {lstm_output.shape}")
            print(f"Logits shape: {logits.shape}")

        return logits


model = CNNLSTMDeepfakeDetector().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE)

model

## Step 6: Debugging Shapes

Run one batch through the model and print the video tensor, CNN feature, LSTM output, and sample prediction shapes before training.

In [ ]:
def debug_one_batch(loader: DataLoader, model: nn.Module):
    if len(loader.dataset) == 0:
        print("No videos available for debugging yet.")
        return

    model.eval()
    videos, labels = next(iter(loader))
    videos = videos.to(device)
    labels = labels.to(device)

    with torch.no_grad():
        logits = model(videos, debug=True)
        probabilities = torch.sigmoid(logits)
        predictions = (probabilities >= 0.5).long()

    print(f"Labels: {labels.detach().cpu().numpy()}")
    print(f"Sample probabilities: {probabilities.detach().cpu().numpy()}")
    print(f"Sample predictions: {predictions.detach().cpu().numpy()}")


debug_one_batch(train_loader, model)

## Step 5: Training

Train with `BCEWithLogitsLoss`, Adam, GPU acceleration when available, and track train loss, validation loss, and validation accuracy.

In [ ]:
def run_train_epoch(model: nn.Module, loader: DataLoader, criterion, optimizer):
    model.train()
    if getattr(model, "freeze_cnn", False):
        model.cnn.eval()

    running_loss = 0.0
    total_samples = 0

    for videos, labels in tqdm(loader, desc="Training", leave=False):
        videos = videos.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(videos)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        batch_size = videos.size(0)
        running_loss += loss.item() * batch_size
        total_samples += batch_size

    return running_loss / max(total_samples, 1)


def run_eval_epoch(model: nn.Module, loader: DataLoader, criterion):
    model.eval()
    running_loss = 0.0
    correct = 0
    total_samples = 0

    with torch.no_grad():
        for videos, labels in tqdm(loader, desc="Validation", leave=False):
            videos = videos.to(device)
            labels = labels.to(device)

            logits = model(videos)
            loss = criterion(logits, labels)
            probabilities = torch.sigmoid(logits)
            predictions = (probabilities >= 0.5).float()

            batch_size = videos.size(0)
            running_loss += loss.item() * batch_size
            correct += (predictions == labels).sum().item()
            total_samples += batch_size

    avg_loss = running_loss / max(total_samples, 1)
    accuracy = correct / max(total_samples, 1)
    return avg_loss, accuracy

In [ ]:
history = []
best_val_loss = float("inf")

if len(train_loader.dataset) == 0 or len(val_loader.dataset) == 0:
    print("Training skipped because train or val split is empty.")
else:
    MODEL_DIR.mkdir(parents=True, exist_ok=True)

    for epoch in range(1, EPOCHS + 1):
        train_loss = run_train_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_accuracy = run_eval_epoch(model, val_loader, criterion)

        history.append(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                "val_loss": val_loss,
                "val_accuracy": val_accuracy,
            }
        )

        print(
            f"Epoch {epoch:02d}/{EPOCHS} | "
            f"train_loss={train_loss:.4f} | "
            f"val_loss={val_loss:.4f} | "
            f"val_accuracy={val_accuracy:.4f}"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "label_map": LABEL_MAP,
                    "sequence_length": SEQUENCE_LENGTH,
                    "image_size": IMAGE_SIZE,
                    "lstm_hidden_size": LSTM_HIDDEN_SIZE,
                    "lstm_num_layers": LSTM_NUM_LAYERS,
                    "feature_dim": model.feature_dim,
                },
                MODEL_PATH,
            )
            print(f"Saved best model to {MODEL_PATH}")

history_df = pd.DataFrame(history)
history_df

## Step 6: Sample Predictions

Inspect a few validation predictions after training.

In [ ]:
def show_sample_predictions(model: nn.Module, loader: DataLoader, max_batches: int = 1):
    if len(loader.dataset) == 0:
        print("No videos available for sample predictions yet.")
        return

    model.eval()
    rows = []

    with torch.no_grad():
        for batch_index, (videos, labels) in enumerate(loader):
            videos = videos.to(device)
            logits = model(videos)
            probabilities = torch.sigmoid(logits).detach().cpu().numpy()
            predictions = (probabilities >= 0.5).astype(int)

            for true_label, probability, prediction in zip(labels.numpy(), probabilities, predictions):
                rows.append(
                    {
                        "true_label": int(true_label),
                        "fake_probability": float(probability),
                        "predicted_label": int(prediction),
                    }
                )

            if batch_index + 1 >= max_batches:
                break

    display(pd.DataFrame(rows))


show_sample_predictions(model, val_loader)

## Step 7: Save Model

Save the final CNN+LSTM model checkpoint to `models/video/cnn_lstm_model.pth`.

In [ ]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "label_map": LABEL_MAP,
        "sequence_length": SEQUENCE_LENGTH,
        "image_size": IMAGE_SIZE,
        "lstm_hidden_size": LSTM_HIDDEN_SIZE,
        "lstm_num_layers": LSTM_NUM_LAYERS,
        "feature_dim": model.feature_dim,
    },
    MODEL_PATH,
)

print(f"Final model saved to: {MODEL_PATH}")